### Understanding feedforward network in Transformer architecture
1. Implementation of linear layer_1 with inputs = feature_dims and outputs = 4 * feature_dims 
2. In this expanded layer, apply `GELU` activation function to introduce non-linearity + tackling vanishing gradient problem with `GELU` or `SIWGELU` activation functions. (** Dimensions will not change in this layer)
3. Implementation of linear layer_2 with inputs = 4 * feature_dims and outputs = feature_dims (Contracting the layers back to original shape)
-------
#### Understanding with diagram
1. Feed Forward Neural layers (Sequential Models) 
- Layer_1: - Expanding the number of neurons for each token. This will help the model to understand pattern. 
- Layer_2: - GELU activation function, preventing vanishing gradient problem. Each neuron will participate in learning 
- Layer_3: - Contracting the layers back to original size as each neuron should have learned the problastic values with each other. 
![Feed-Forward Layer](../Images/image_2.png)

In [5]:
import torch 
import torch.nn as nn
import math

In [8]:
# Understanding GELU activation function with an example 
inputs = torch.randn([2, 3, 6]) # num_batch, tokens, feature_dims 

# Using gelu approximation formula to calculate each element computation 
gelu = 0.5 * inputs * (1 + torch.tanh(
    torch.sqrt(torch.tensor(2 / torch.pi)) * 
    (inputs + 0.44715 * torch.pow(inputs, 3))
))
print(f'After Activation: Tensor.value \n {gelu}')

After Activation: Tensor.value 
 tensor([[[-1.4852e-01,  1.4856e+00,  7.7666e-01,  1.0224e+00,  3.8122e-01,
          -1.6144e-03],
         [ 5.0438e-01, -1.4846e-01, -2.0953e-02, -1.4389e-01, -1.0779e-01,
           1.9617e+00],
         [-8.8380e-02, -1.4810e-01, -9.7981e-02,  9.4716e-03, -3.2176e-02,
          -1.5334e-02]],

        [[-5.9746e-07, -2.2609e-02,  5.4010e-01,  3.1843e-02, -1.4854e-01,
          -6.9834e-04],
         [-6.1402e-02,  9.5462e-02, -4.1981e-02, -7.2709e-02,  1.8466e+00,
          -6.5872e-02],
         [ 2.2408e+00, -1.4462e-01, -1.2666e-01,  2.2759e+00,  4.5098e-01,
          -1.4048e-01]]])


In [13]:
# Converting above logic into modular code format (classes)
class GELU(nn.Module):
    def __init__(self):
        super().__init__() # calling initialize method for nn.module 
    
    def forward(self, x):
        # Implementation of GELU activation. No change in shape of the input tensor matrices 
        # Using gelu closest approximate function, 
        # as calculatation with cdf of gaussian (error rate method) is computatation heavy in GPU architecture. 
        gelu_calculate = 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2 / torch.pi)) * 
            (x + 0.44715 * torch.pow(x, 3))
        ))

        return gelu_calculate

In [14]:
torch.manual_seed(123) # Seed to run for 123 time until generating next set of random values  
inputs = torch.randn([2, 3, 6]) # num_batch, tokens, feature_dims 
gelu = GELU() # feature_dims = 6
print(f'After GELLU: Tensor.value \n{gelu(inputs)}')

After GELLU: Tensor.value 
tensor([[[ 2.1517e-01, -7.6189e-02, -1.1429e-01, -1.4869e-01,  2.2400e-01,
           5.1433e-01],
         [-9.0366e-02, -1.3057e-01,  6.3238e-01, -5.0770e-02,  5.5543e-01,
          -1.9852e-02],
         [ 1.0262e-01,  1.8944e+00,  3.4993e-01,  1.6397e-01, -3.6140e-02,
          -8.5915e-02]],

        [[-7.3016e-02,  8.0991e-01, -1.3222e-01, -1.0441e-01, -1.4580e-01,
          -7.1732e-02],
         [ 8.6396e-01,  1.4972e-01, -1.4464e-01, -1.0697e-04,  5.3004e-02,
          -4.3258e-02],
         [ 1.6182e-01, -1.4870e-01, -1.2343e-01, -1.4437e-01, -9.2803e-02,
          -3.3360e-03]]])


In [15]:
# Understanding + Coding: Feed-forward layer in Transformer architecture 
class FeedForwardExample(nn.Module):
    def __init__(self, feature_dims: int):
        super().__init__() # Importing nn.Module class members 
        self.feature_dims = feature_dims
        # Stacking up layers sequentially 
        self.feed_forward = nn.Sequential(
            nn.Linear(feature_dims, 4 * feature_dims), # Layer_1: Expansion 
            GELU(), # Activation Function 
            nn.Linear(4 * feature_dims, feature_dims) # Layer_2: Contraction 
        )
    
    def forward(self, x):
        # calling out feed_forward netword of sequential model 
        return self.feed_forward(x)
    

In [18]:
torch.manual_seed(123) # Seed to run for 123 time until generating next set of random values  
inputs = torch.randn([2, 3, 768]) # num_batch, tokens, feature_dims 
feature_dims = inputs.shape[-1]
ffn = FeedForwardExample(feature_dims)
print(f'Feed_forward: \n{ffn(inputs)}')

Feed_forward: 
tensor([[[ 0.4417, -0.2622,  0.2850,  ..., -0.1988, -0.0270,  0.0340],
         [ 0.2268,  0.0194,  0.1025,  ...,  0.1553,  0.0518,  0.0901],
         [ 0.0641, -0.6159,  0.1624,  ..., -0.1067,  0.2178, -0.2541]],

        [[ 0.0936, -0.1580,  0.2364,  ...,  0.0525,  0.1404, -0.0391],
         [ 0.1164, -0.3707, -0.0959,  ..., -0.1683,  0.0897,  0.0548],
         [ 0.4430, -0.0238,  0.0260,  ...,  0.2004,  0.0734,  0.0372]]],
       grad_fn=<ViewBackward0>)
